# trying to analyze the data
main idea:
The analysis should answer three questions in order:

1. Where does Migros currently have strong or weak coverage?
2. Which populated areas have no Migros?
3. Which of those areas have enough population and purchasing power to justify a new store?

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

In [2]:
from pathlib import Path

plot_folder = Path("plots")
plot_folder.mkdir(parents=True, exist_ok=True)

In [3]:
# Load and inspect the master data

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

analysis_df = pd.read_csv("data/migros_master_data.csv", dtype={"postal_code": "string"})

print("Shape:", analysis_df.shape)
display(analysis_df.head())

Shape: (3176, 19)


,postal_code,population,avg_income_per_taxpayer,municipality_name,canton_code,aldi_count,coop_count,denner_count,lidl_count,migros_count,external_competitor_count,migros_group_count,postcode_latitude,postcode_longitude,locality_name,complete_location_data,has_migros,population_per_migros,primary_canton
0,1000,4248,57188.176061,Lausanne,VD,0,0,0,0,0,0,0,46.552076,6.686622,Lausanne 25,True,False,4248.000000,VD
1,1003,6879,57188.176061,Lausanne,VD,2,1,1,2,2,5,3,46.520577,6.631877,Lausanne,True,True,3439.500000,VD
2,1004,31463,57188.176061,Lausanne,VD,1,5,1,0,3,6,4,46.528291,6.618649,Lausanne,True,True,10487.666667,VD
3,1005,12454,57188.176061,Lausanne,VD,0,3,0,0,0,3,0,46.520897,6.642754,Lausanne,True,False,12454.000000,VD
4,1006,15621,57188.176061,Lausanne,VD,0,2,2,0,1,2,3,46.512854,6.633803,Lausanne,True,True,15621.000000,VD


In [4]:
# inspect data
print(analysis_df.columns.tolist())
print()
analysis_df.info()

['postal_code', 'population', 'avg_income_per_taxpayer', 'municipality_name', 'canton_code', 'aldi_count', 'coop_count', 'denner_count', 'lidl_count', 'migros_count', 'external_competitor_count', 'migros_group_count', 'postcode_latitude', 'postcode_longitude', 'locality_name', 'complete_location_data', 'has_migros', 'population_per_migros', 'primary_canton']

<class 'pandas.DataFrame'>
RangeIndex: 3176 entries, 0 to 3175
Data columns (total 19 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   postal_code                3176 non-null   string 
 1   population                 3176 non-null   int64  
 2   avg_income_per_taxpayer    3170 non-null   float64
 3   municipality_name          3170 non-null   str    
 4   canton_code                3170 non-null   str    
 5   aldi_count                 3176 non-null   int64  
 6   coop_count                 3176 non-null   int64  
 7   denner_count               3176 n

In [5]:
analysis_df.isna().sum()

postal_code                  0
population                   0
avg_income_per_taxpayer      6
municipality_name            6
canton_code                  6
aldi_count                   0
coop_count                   0
denner_count                 0
lidl_count                   0
migros_count                 0
external_competitor_count    0
migros_group_count           0
postcode_latitude            6
postcode_longitude           6
locality_name                6
complete_location_data       0
has_migros                   0
population_per_migros        0
primary_canton               6
dtype: int64

In [6]:
# Check that every row represents one postcode:
print("Rows:", len(analysis_df))
print(
    "Unique postcodes:",
    analysis_df["postal_code"].nunique()
)

print(
    "Duplicate postcodes:",
    analysis_df["postal_code"].duplicated().sum()
)
# Check total population:
print(
    "Total population:",
    f"{analysis_df['population'].sum():,}"
)

#Check store totals:
store_columns = [
    "migros_count",
    "coop_count",
    "denner_count",
    "aldi_count",
    "lidl_count"
]

analysis_df[store_columns].sum()

Rows: 3176
Unique postcodes: 3176
Duplicate postcodes: 0
Total population: 9,127,125


migros_count    717
coop_count      923
denner_count    724
aldi_count      240
lidl_count      191
dtype: int64

In [7]:
# Exclude only the six postcodes with incomplete income and location data
scoring_df = analysis_df[analysis_df["complete_location_data"] == True].copy()

print("Full dataset:", analysis_df.shape)
print("Scoring dataset:", scoring_df.shape)

Full dataset: (3176, 19)
Scoring dataset: (3170, 19)


In [8]:
analysis_df["complete_location_data"].value_counts(dropna=False)

complete_location_data
True     3170
False       6
Name: count, dtype: int64

In [9]:
scoring_df.isna().sum()

postal_code                  0
population                   0
avg_income_per_taxpayer      0
municipality_name            0
canton_code                  0
aldi_count                   0
coop_count                   0
denner_count                 0
lidl_count                   0
migros_count                 0
external_competitor_count    0
migros_group_count           0
postcode_latitude            0
postcode_longitude           0
locality_name                0
complete_location_data       0
has_migros                   0
population_per_migros        0
primary_canton               0
dtype: int64

# Calculate initial project KPIs

In [10]:
total_population = analysis_df["population"].sum()
total_migros = analysis_df["migros_count"].sum()

postcodes_with_migros = (analysis_df["migros_count"] > 0).sum()

postcodes_without_migros = (analysis_df["migros_count"] == 0).sum()

population_without_migros = analysis_df.loc[analysis_df["migros_count"] == 0,"population"].sum()

population_without_migros_pct = (population_without_migros/ total_population * 100)

print("Total Swiss population:", f"{total_population:,}")
print("Total Migros stores:", total_migros)
print("Postcodes with Migros:", postcodes_with_migros)
print("Postcodes without Migros:", postcodes_without_migros)
print(
    "Population in postcodes without Migros:",
    f"{population_without_migros:,}"
)
print(
    "Population share without Migros:",
    f"{population_without_migros_pct:.2f}%"
)

Total Swiss population: 9,127,125
Total Migros stores: 717
Postcodes with Migros: 549
Postcodes without Migros: 2627
Population in postcodes without Migros: 3,741,584
Population share without Migros: 40.99%


In [11]:
# small summary table - stores by brand
brand_counts_df = pd.DataFrame({
    "brand": [
        "Migros",
        "Coop",
        "Denner",
        "Aldi",
        "Lidl"
    ],
    "number_of_stores": [
        analysis_df["migros_count"].sum(),
        analysis_df["coop_count"].sum(),
        analysis_df["denner_count"].sum(),
        analysis_df["aldi_count"].sum(),
        analysis_df["lidl_count"].sum()
    ]
})

brand_counts_df = brand_counts_df.sort_values(
    "number_of_stores",
    ascending=True
)

brand_counts_df

,brand,number_of_stores
4,Lidl,191
3,Aldi,240
0,Migros,717
2,Denner,724
1,Coop,923


In [12]:
# plotting
brand_colors = {
    "Migros": "#F58220",
    "Coop": "#E30613",
    "Denner": "#19A831",
    "Aldi": "#120DA9",
    "Lidl": "#8B0E70"
}

fig_brand_counts = px.bar(
    brand_counts_df,
    x="number_of_stores",
    y="brand",
    orientation="h",
    color="brand",
    color_discrete_map=brand_colors,
    text="number_of_stores",
    title="Supermarket Locations by Brand in Switzerland"
)

fig_brand_counts.update_traces(
    textposition="outside",
    hovertemplate=(
        "<b>%{y}</b><br>"
        "Stores: %{x:,}"
        "<extra></extra>"
    )
)

fig_brand_counts.update_layout(
    showlegend=False,
    xaxis_title="Number of supermarket locations",
    yaxis_title="Brand",
    template="plotly_white",
    height=450,
    width=900,
    margin=dict(l=80, r=80, t=80, b=60)
)
fig_brand_counts.update_xaxes(tickformat=",")
fig_brand_counts.show()

In [13]:
# Identify postcodes without Migros
no_migros_df = scoring_df[scoring_df["migros_count"] == 0].copy()

# since there was 2 Geneva 
# Create a unique chart label using locality and postcode
no_migros_df["location_label"] = no_migros_df["locality_name"] + " (" + no_migros_df["postal_code"] + ")"

no_migros_df = no_migros_df.sort_values("population",ascending=False)

display(
    no_migros_df[
        [
            "postal_code",
            "locality_name",
            "municipality_name",
            "canton_code",
            "population",
            "avg_income_per_taxpayer",
            "external_competitor_count"
        ]
    ].head(20)
)

,postal_code,locality_name,municipality_name,canton_code,population,avg_income_per_taxpayer,external_competitor_count
129,1202,Genève,"Genève, Pregny-Chambésy",GE,36246,59153.025471,3
570,1920,Martigny,"Martigny, Martigny-Combe, Salvan, Vernayaz",VS,20005,56663.734660,1
150,1226,Thônex,Thônex,GE,16940,62667.582916,1
135,1208,Genève,Genève,GE,16379,58950.440219,0
203,1290,Versoix,"Chavannes-des-Bois, Collex-Bossy, Mies, Versoix","GE, VD",15164,87364.782045,2
3,1005,Lausanne,Lausanne,VD,12454,57188.176061,3
856,3008,Bern,Bern,BE,11463,63636.523330,4
2916,8854,Siebnen,"Galgenen, Schübelbach, Wangen (SZ)",SZ,11369,76149.128060,3
155,1233,Bernex,Bernex,GE,11115,74411.860731,1
38,1052,Le Mont-sur-Lausanne,Le Mont-sur-Lausanne,VD,9790,80137.317266,2


In [14]:
# cleaning the data further
# SELECT THE TOP 15 POPULATION GAPS
top_15_no_migros = no_migros_df.nlargest(15, "population").copy()
top_15_no_migros = top_15_no_migros.sort_values("population", ascending=True).reset_index(drop=True)

display(
    top_15_no_migros[
        [
            "postal_code",
            "locality_name",
            "municipality_name",
            "canton_code",
            "population",
            "avg_income_per_taxpayer",
            "external_competitor_count"
        ]
    ]
)


,postal_code,locality_name,municipality_name,canton_code,population,avg_income_per_taxpayer,external_competitor_count
0,4663,Aarburg,"Aarburg, Boningen, Olten","AG, SO",9049,63019.293937,0
1,5036,Oberentfelden,Oberentfelden,AG,9061,63550.554636,2
2,8623,Wetzikon ZH,"Pfäffikon, Wetzikon (ZH)",ZH,9092,68251.227040,0
3,3097,Liebefeld,"Bern, Köniz",BE,9174,69570.297774,1
4,8802,Kilchberg ZH,Kilchberg (ZH),ZH,9571,141795.142249,1
5,1052,Le Mont-sur-Lausanne,Le Mont-sur-Lausanne,VD,9790,80137.317266,2
6,1233,Bernex,Bernex,GE,11115,74411.860731,1
7,8854,Siebnen,"Galgenen, Schübelbach, Wangen (SZ)",SZ,11369,76149.128060,3
8,3008,Bern,Bern,BE,11463,63636.523330,4
9,1005,Lausanne,Lausanne,VD,12454,57188.176061,3


In [15]:
# Create the first opportunity chart
#top_population_gaps = no_migros_df.nlargest(15,"population").sort_values("population", ascending=True)

fig_population_gaps = px.bar(
    top_15_no_migros,
    x="population",
    y="location_label",
    orientation="h",
    color="avg_income_per_taxpayer",
    color_continuous_scale="Oranges",
    text="population",
    hover_data={
        "postal_code": True,
        "locality_name": False,
        "location_label": False,
        "municipality_name": True,
        "canton_code": True,
        "population": ":,",
        "avg_income_per_taxpayer": ":,.0f",
        "external_competitor_count": True
    },
    labels={
        "population": "Population",
        "location_label": "Locality and postcode",
        "municipality_name": "Municipality",
        "canton_code": "Canton",
        "avg_income_per_taxpayer": "Average income per taxpayer",
        "external_competitor_count": "External competitors"
    },
    title="Largest Population Centres Without a Migros Supermarket"
)

fig_population_gaps.update_traces(
    texttemplate="%{text:,.0f}",
    textposition="outside",
    cliponaxis=False
)

fig_population_gaps.update_layout(
    xaxis_title="Population",
    yaxis_title="Locality and postcode",
    template="plotly_white",
    height=650,
    width=1050,
    coloraxis_colorbar_title="Average income<br>per taxpayer",
    margin=dict(l=160, r=100, t=80, b=60)
)

fig_population_gaps.update_xaxes(tickformat=",")
fig_population_gaps.show()

# Area wise
Switzerland
   ↓
Canton
   ↓
Municipality
   ↓
Locality
   ↓
Postal-code area

# Canton based
Let’s first aggregate the postcode-level data into cantons and build two interactive Plotly charts:

1. Number of Migros stores in each canton.
2. Percentage of Migros versus direct competitors in every canton. 
Migros = Migros supermarkets
Competitors = Coop + Aldi + Lidl

In [16]:
# Some postcodes cover more than one canton. Use the municipality with the largest postcode weight to determine the primary canton.
# Assign every postcode to its main canton
# Confirm that the primary canton column exists in the master data
print("Missing primary cantons:", analysis_df["primary_canton"].isna().sum())
print("Number of cantons:", analysis_df["primary_canton"].nunique())

Missing primary cantons: 6
Number of cantons: 26


In [17]:
## Reuse rows with complete income, municipality, canton and coordinate data
canton_analysis_df = scoring_df.copy()

In [18]:
print("Canton analysis rows:", len(canton_analysis_df))
print("Missing primary cantons:", canton_analysis_df["primary_canton"].isna().sum())
print("Number of cantons:", canton_analysis_df["primary_canton"].nunique())

Canton analysis rows: 3170
Missing primary cantons: 0
Number of cantons: 26


In [19]:
# Add store counts across all postcodes belonging to each canton
canton_store_df = canton_analysis_df.groupby("primary_canton", as_index=False)[["migros_count", "coop_count", "denner_count", "aldi_count", "lidl_count"]].sum()
canton_store_df.head()

,primary_canton,migros_count,coop_count,denner_count,aldi_count,lidl_count
0,AG,42,56,49,15,16
1,AI,1,1,1,0,0
2,AR,4,4,4,1,1
3,BE,72,138,84,23,18
4,BL,25,29,23,6,4


In [20]:
# Direct external competitors are Coop, Aldi and Lidl
canton_store_df["competitor_count"] = canton_store_df["coop_count"] + canton_store_df["aldi_count"] + canton_store_df["lidl_count"]
canton_store_df.head()

,primary_canton,migros_count,coop_count,denner_count,aldi_count,lidl_count,competitor_count
0,AG,42,56,49,15,16,87
1,AI,1,1,1,0,0,1
2,AR,4,4,4,1,1,6
3,BE,72,138,84,23,18,179
4,BL,25,29,23,6,4,39


In [21]:
# total Migros group count
# Denner belongs to Migros Group
canton_store_df["migros_group_count"] = canton_store_df["migros_count"] + canton_store_df["denner_count"]
print(canton_store_df[["migros_count", "coop_count", "denner_count", "aldi_count", "lidl_count"]].sum())

migros_count    717
coop_count      923
denner_count    724
aldi_count      240
lidl_count      191
dtype: int64


In [22]:
print(canton_store_df[["migros_count", "denner_count", "migros_group_count", "competitor_count"]].sum())

migros_count           717
denner_count           724
migros_group_count    1441
competitor_count      1354
dtype: int64


In [23]:
#plot migros stores by canton
# Sort cantons so the canton with the most Migros stores appears at the top
migros_by_canton = canton_store_df.sort_values("migros_count", ascending=True)

In [24]:
# Create an interactive horizontal bar chart
fig_migros_canton = px.bar(
    migros_by_canton,
    x="migros_count",
    y="primary_canton",
    orientation="h",
    color="migros_count",
    color_continuous_scale="Oranges",
    text="migros_count",
    hover_data={
        "migros_count": ":,",
        "denner_count": ":,",
        "coop_count": ":,",
        "aldi_count": ":,",
        "lidl_count": ":,",
        "competitor_count": ":,"
    },
    labels={
        "primary_canton": "Canton",
        "migros_count": "Migros stores",
        "denner_count": "Denner stores",
        "coop_count": "Coop stores",
        "aldi_count": "Aldi stores",
        "lidl_count": "Lidl stores",
        "competitor_count": "External competitor stores"
    },
    title="Number of Migros Supermarkets by Canton"
)

In [25]:
# Display the store count outside each bar
fig_migros_canton.update_traces(textposition="outside", cliponaxis=False)

In [26]:
# Format the chart
fig_migros_canton.update_layout(template="plotly_white", height=750, width=1000, xaxis_title="Number of Migros stores", yaxis_title="Canton", coloraxis_showscale=False, margin=dict(l=80, r=80, t=80, b=60))#
fig_migros_canton.show()

# Will Burtin based visualization
Each sector = one canton
Radial bar length = canton population
Bar colour = average income per taxpayer
Orange bubble size = number of Migros stores
Hover = exact population, income, Migros and competitor counts


In [27]:
# Create a copy containing rows with complete data
canton_visual_df = scoring_df.copy()

# Calculate population-weighted income for each postcode
canton_visual_df["income_x_population"] = canton_visual_df["avg_income_per_taxpayer"] * canton_visual_df["population"]

canton_visual_df.head()


,postal_code,population,avg_income_per_taxpayer,municipality_name,canton_code,aldi_count,coop_count,denner_count,lidl_count,migros_count,external_competitor_count,migros_group_count,postcode_latitude,postcode_longitude,locality_name,complete_location_data,has_migros,population_per_migros,primary_canton,income_x_population
0,1000,4248,57188.176061,Lausanne,VD,0,0,0,0,0,0,0,46.552076,6.686622,Lausanne 25,True,False,4248.000000,VD,2.429354e+08
1,1003,6879,57188.176061,Lausanne,VD,2,1,1,2,2,5,3,46.520577,6.631877,Lausanne,True,True,3439.500000,VD,3.933975e+08
2,1004,31463,57188.176061,Lausanne,VD,1,5,1,0,3,6,4,46.528291,6.618649,Lausanne,True,True,10487.666667,VD,1.799312e+09
3,1005,12454,57188.176061,Lausanne,VD,0,3,0,0,0,3,0,46.520897,6.642754,Lausanne,True,False,12454.000000,VD,7.122215e+08
4,1006,15621,57188.176061,Lausanne,VD,0,2,2,0,1,2,3,46.512854,6.633803,Lausanne,True,True,15621.000000,VD,8.933365e+08


In [28]:

# Aggregate population, income and store counts for every canton
canton_wheel_df = canton_visual_df.groupby("primary_canton", as_index=False).agg(population=("population", "sum"), income_x_population=("income_x_population", "sum"), migros_count=("migros_count", "sum"), coop_count=("coop_count", "sum"), denner_count=("denner_count", "sum"), aldi_count=("aldi_count", "sum"), lidl_count=("lidl_count", "sum"))

#Calculate population-weighted average income:
canton_wheel_df["avg_income_per_taxpayer"] = canton_wheel_df["income_x_population"] / canton_wheel_df["population"]

canton_wheel_df.head()

,primary_canton,population,income_x_population,migros_count,coop_count,denner_count,aldi_count,lidl_count,avg_income_per_taxpayer
0,AG,743776,5.260829e+10,42,56,49,15,16,70731.362717
1,AI,16524,7.478011e+08,1,1,1,0,0,45255.452062
2,AR,56680,3.794412e+09,4,4,4,1,1,66944.463222
3,BE,1068969,6.821139e+10,72,138,84,23,18,63810.449892
4,BL,303824,2.296669e+10,25,29,23,6,4,75592.077105


In [29]:
#Calculate direct competitor count
canton_wheel_df["competitor_count"] = canton_wheel_df["coop_count"] + canton_wheel_df["aldi_count"] + canton_wheel_df["lidl_count"]

#Sort the cantons by population:
canton_wheel_df = canton_wheel_df.sort_values("population", ascending=False).reset_index(drop=True)

canton_wheel_df.head()

,primary_canton,population,income_x_population,migros_count,coop_count,denner_count,aldi_count,lidl_count,avg_income_per_taxpayer,competitor_count
0,ZH,1643889,1.317515e+11,122,162,121,50,31,80146.221882,243
1,BE,1068969,6.821139e+10,72,138,84,23,18,63810.449892,179
2,VD,876568,6.077177e+10,69,73,74,17,17,69329.216368,107
3,AG,743776,5.260829e+10,42,56,49,15,16,70731.362717,87
4,SG,536154,3.453994e+10,51,39,42,20,14,64421.684291,73


In [30]:
#Transform population for the radial layout
# The transformation keeps small cantons visible while preserving their order.
population_min = canton_wheel_df["population"].min()
population_max = canton_wheel_df["population"].max()
canton_wheel_df["population_radius"] = 35 + 60 * (canton_wheel_df["population"] - population_min) / (population_max - population_min)
#This maps canton population onto a visible radial range from 35 to 95. The exact population remains available in the tooltip.

In [31]:
# #Create the Burtin-inspired chart
# fig_canton_wheel = go.Figure()

# #Add population bars coloured by income:
# fig_canton_wheel.add_trace(
#     go.Barpolar(
#         r=canton_wheel_df["population_radius"],
#         theta=canton_wheel_df["primary_canton"],
#         width=[11] * len(canton_wheel_df),
#         marker=dict(
#             color=canton_wheel_df["avg_income_per_taxpayer"],
#             colorscale="YlOrRd",
#             colorbar=dict(title="Average income<br>per taxpayer"),
#             line=dict(color="white", width=1)
#         ),
#         customdata=canton_wheel_df[
#             [
#                 "population",
#                 "avg_income_per_taxpayer",
#                 "migros_count",
#                 "competitor_count",
#                 "denner_count"
#             ]
#         ],
#         hovertemplate=(
#             "<b>Canton %{theta}</b><br>"
#             "Population: %{customdata[0]:,.0f}<br>"
#             "Average income: CHF %{customdata[1]:,.0f}<br>"
#             "Migros stores: %{customdata[2]:,.0f}<br>"
#             "External competitors: %{customdata[3]:,.0f}<br>"
#             "Denner stores: %{customdata[4]:,.0f}"
#             "<extra></extra>"
#         ),
#         name="Population and income"
#     )
# )

In [32]:
# # Calculate the bubble-size scaling
# bubble_reference = 2 * canton_wheel_df["migros_count"].max() / 35**2

# #Add Migros bubbles
# fig_canton_wheel.add_trace(
#     go.Scatterpolar(
#         r=canton_wheel_df["population_radius"] + 5,
#         theta=canton_wheel_df["primary_canton"],
#         mode="markers",
#         marker=dict(
#             size=canton_wheel_df["migros_count"],
#             sizemode="area",
#             sizeref=bubble_reference,
#             sizemin=5,
#             color="#3118D3",
#             opacity=0.9,
#             line=dict(color="white", width=1)
#         ),
#         customdata=canton_wheel_df[["migros_count", "population", "avg_income_per_taxpayer"]],
#         hovertemplate=(
#             "<b>Canton %{theta}</b><br>"
#             "Migros stores: %{customdata[0]:,.0f}<br>"
#             "Population: %{customdata[1]:,.0f}<br>"
#             "Average income: CHF %{customdata[2]:,.0f}"
#             "<extra></extra>"
#         ),
#         name="Migros stores"
#     )
# )

In [33]:
# # Format the wheel:
# fig_canton_wheel.update_layout(
#     title="Swiss Canton Market Landscape",
#     template="plotly_white",
#     height=850,
#     width=950,
#     polar=dict(
#         radialaxis=dict(visible=False, range=[0, 110]),
#         angularaxis=dict(direction="clockwise", rotation=90)
#     ),
#     legend=dict(orientation="h", x=0.5, xanchor="center", y=-0.05),
#     margin=dict(l=70, r=120, t=100, b=80)
# )
# fig_canton_wheel.show()

# trying a different chart
bubble quadrant chart

It will show:

X-axis: average income per taxpayer
Y-axis: Migros stores per 100,000 residents
Bubble size: canton population
Bubble colour: competitor stores per 100,000 residents
Bubble label: canton code
-- High income + low Migros coverage

In [34]:
canton_names = {
    "AG": "Aargau",
    "AI": "Appenzell Innerrhoden",
    "AR": "Appenzell Ausserrhoden",
    "BE": "Bern",
    "BL": "Basel-Landschaft",
    "BS": "Basel-Stadt",
    "FR": "Fribourg",
    "GE": "Geneva",
    "GL": "Glarus",
    "GR": "Graubünden",
    "JU": "Jura",
    "LU": "Lucerne",
    "NE": "Neuchâtel",
    "NW": "Nidwalden",
    "OW": "Obwalden",
    "SG": "St. Gallen",
    "SH": "Schaffhausen",
    "SO": "Solothurn",
    "SZ": "Schwyz",
    "TG": "Thurgau",
    "TI": "Ticino",
    "UR": "Uri",
    "VD": "Vaud",
    "VS": "Valais",
    "ZG": "Zug",
    "ZH": "Zürich"
}

In [35]:
canton_wheel_df["canton_name"] = canton_wheel_df["primary_canton"].map(canton_names)

In [36]:
# Calculate Migros store density per 100,000 residents
canton_wheel_df["migros_per_100k"] = canton_wheel_df["migros_count"] / canton_wheel_df["population"] * 100000

In [37]:
# Calculate external competitor density per 100,000 residents
canton_wheel_df["competitors_per_100k"] = canton_wheel_df["competitor_count"] / canton_wheel_df["population"] * 100000

In [38]:
#Calculate median values to divide the chart into quadrants
median_income = canton_wheel_df["avg_income_per_taxpayer"].median()
median_migros_coverage = canton_wheel_df["migros_per_100k"].median()

In [39]:
# Create the interactive bubble chart
fig_canton_bubble = px.scatter(
    canton_wheel_df,
    x="avg_income_per_taxpayer",
    y="migros_per_100k",
    size="population",
    color="competitors_per_100k",
    text="primary_canton",
    hover_name="canton_name",
    size_max=80,
    color_continuous_scale="Turbo",
    custom_data=["canton_name", "population", "avg_income_per_taxpayer", "migros_count", "competitor_count", "migros_per_100k", "competitors_per_100k", "denner_count"],
    # hover_data={
    #     "primary_canton": False,
    #     "canton_name": False,
    #     "population": ":,",
    #     "avg_income_per_taxpayer": ":,.0f",
    #     "migros_count": ":,",
    #     "migros_per_100k": ":.2f",
    #     "competitor_count": ":,",
    #     "competitors_per_100k": ":.2f",
    #     "denner_count": ":,"
    # },
    labels={
        "avg_income_per_taxpayer": "Average income per taxpayer (CHF)",
        "migros_per_100k": "Migros stores per 100,000 residents",
        "competitors_per_100k": "Competitors per 100,000 residents",
        "population": "Population",
        "migros_count": "Migros stores",
        "competitor_count": "External competitors",
        "denner_count": "Denner stores"
    },
    title="Migros Market Coverage and Purchasing Power by Canton"
)

In [40]:
fig_canton_bubble.update_traces(
    hovertemplate=(
        "<b>%{customdata[0]}</b><br>"
        "Population: %{customdata[1]:,.0f}<br>"
        "Average income per taxpayer: CHF %{customdata[2]:,.0f}<br>"
        "Total Migros stores: %{customdata[3]:,.0f}<br>"
        "Total competitor stores: %{customdata[4]:,.0f}<br>"
        "Migros per 100,000 residents: %{customdata[5]:.2f}<br>"
        "Competitors per 100,000 residents: %{customdata[6]:.2f}<br>"
        "Denner stores: %{customdata[7]:,.0f}"
        "<extra></extra>"
    )
)

In [41]:
# Add the median reference lines
fig_canton_bubble.add_vline(x=median_income, line_width=1, line_dash="dash", line_color="grey")

fig_canton_bubble.add_hline(y=median_migros_coverage, line_width=1, line_dash="dash", line_color="grey")

#Format the bubbles and labels
#fig_canton_bubble.update_traces(textposition="middle center", textfont=dict(color="black", size=12), marker=dict(opacity=0.9, line=dict(color="white", width=1.5)))
fig_canton_bubble.update_traces(textposition="middle center", textfont=dict(color="#04101E", size=11), marker=dict(opacity=0.82, line=dict(color="grey", width=1.5)))
#format the chart
fig_canton_bubble.update_layout(
    title=dict(text="Migros Market Coverage and Purchasing Power by Canton", x=0.5, xanchor="center", y=0.97, yanchor="top", font=dict(size=24)),
    template="plotly_white",
    height=750,
    width=1200,
    xaxis=dict(title=dict(text="Average income per taxpayer (CHF)", font=dict(size=18)), tickfont=dict(size=14)),
    yaxis=dict(title=dict(text="Migros stores per 100K residents", font=dict(size=18)), tickfont=dict(size=14)),
    coloraxis_colorbar_title="Competitors<br>per 100K",
    margin=dict(l=90, r=130, t=90, b=80),
    paper_bgcolor="#DBD5D5",
    plot_bgcolor="#DBD5D5"
)

fig_canton_bubble.update_xaxes(tickprefix="CHF ", tickformat=",", gridcolor="rgba(150,150,150,0.2)")
fig_canton_bubble.update_yaxes(tickformat=".1f", gridcolor="rgba(150,150,150,0.2)")
#fig_canton_bubble.show()

In [42]:
# Add quadrant descriptions
x_min = canton_wheel_df["avg_income_per_taxpayer"].min()
x_max = canton_wheel_df["avg_income_per_taxpayer"].max()
y_min = canton_wheel_df["migros_per_100k"].min()
y_max = canton_wheel_df["migros_per_100k"].max()


In [43]:
fig_canton_bubble.update_layout(annotations=[])


In [44]:
x_left_center = (x_min + median_income) / 2
x_right_center = (median_income + x_max) / 2
y_lower_center = (y_min + median_migros_coverage) / 2
y_upper_center = (median_migros_coverage + y_max) / 2


In [45]:
# Upper-left quadrant
#fig_canton_bubble.add_annotation(x=x_max, y=y_min, text="Potential opportunity<br>High income, weak Migros coverage", showarrow=False, xanchor="right", yanchor="bottom", bgcolor="rgba(245,130,32,0.15)")
#fig_canton_bubble.add_annotation(x=1.5, y=1.5, xref="paper", yref="paper", text="Coverage review<br>Lower income, strong coverage", showarrow=False, xanchor="left", bgcolor="rgba(50, 110, 190, 0.16)", bordercolor="rgba(50, 110, 190, 0.55)", borderwidth=1, borderpad=5)
fig_canton_bubble.add_shape(type="rect", x0=x_min, x1=median_income, y0=median_migros_coverage, y1=y_max, fillcolor="#4443B0", opacity=0.18, line_width=0, layer="below")
fig_canton_bubble.add_annotation(x=x_left_center, y=y_upper_center, text="<b>COVERAGE REVIEW</b>", showarrow=False, font=dict(size=20, color="#6260CA"), opacity=0.6)


# Upper-right quadrant
#fig_canton_bubble.add_annotation(x=x_max, y=y_max, text="Strong affluent presence<br>High income, strong coverage", showarrow=False, xanchor="right", yanchor="top", bgcolor="rgba(80,160,100,0.15)")
#fig_canton_bubble.add_annotation(x=0.99, y=1.03, xref="paper", yref="paper", text="Strong affluent presence<br>High income, strong coverage", showarrow=False, xanchor="right", bgcolor="rgba(46, 160, 67, 0.18)", bordercolor="rgba(46, 160, 67, 0.6)", borderwidth=1, borderpad=5)
fig_canton_bubble.add_shape(type="rect", x0=median_income, x1=x_max, y0=median_migros_coverage, y1=y_max, fillcolor="#2C854B", opacity=0.20, line_width=0, layer="below")
fig_canton_bubble.add_annotation(x=x_right_center, y=y_upper_center, text="<b>STRONG AFFLUENT PRESENCE</b>", showarrow=False, font=dict(size=20, color="#319C56"), opacity=0.6)

# Lower-left quadrant
#fig_canton_bubble.add_annotation(x=0.01, y=1, xref="paper", yref="paper", text="Lower-priority market<br>Lower income, weak coverage", showarrow=False, xanchor="left", bgcolor="rgba(120, 120, 120, 0.15)", bordercolor="rgba(120, 120, 120, 0.5)", borderwidth=1, borderpad=5)
#fig_canton_bubble.add_annotation(x=x_min, y=y_min, text="Lower-priority market<br>Lower income, weak coverage", showarrow=False, xanchor="left", yanchor="bottom", bgcolor="rgba(150,150,150,0.12)")
fig_canton_bubble.add_shape(type="rect", x0=x_min, x1=median_income, y0=y_min, y1=median_migros_coverage, fillcolor="#ACA8A8", opacity=0.20, line_width=0, layer="below")
fig_canton_bubble.add_annotation(x=x_left_center, y=y_lower_center, text="<b>LOWER PRIORITY</b>", showarrow=False, font=dict(size=20, color="#8E8B8B"), opacity=0.6)

# Lower-right quadrant
#fig_canton_bubble.add_annotation(x=0.99, y=-0.14, xref="paper", yref="paper", text="Potential opportunity<br>High income, weak Migros coverage", showarrow=False, xanchor="right", bgcolor="rgba(245, 130, 32, 0.18)", bordercolor="rgba(245, 130, 32, 0.6)", borderwidth=1, borderpad=5)
#fig_canton_bubble.add_annotation(x=x_min, y=y_max, text="Coverage review<br>Lower income, strong coverage", showarrow=False, xanchor="left", yanchor="top", bgcolor="rgba(70,120,180,0.12)")
fig_canton_bubble.add_shape(type="rect", x0=median_income, x1=x_max, y0=y_min, y1=median_migros_coverage, fillcolor="#D42222", opacity=0.20, line_width=0, layer="below")
fig_canton_bubble.add_annotation(x=x_right_center, y=y_lower_center, text="<b>POTENTIAL OPPORTUNITY</b>", showarrow=False, font=dict(size=20, color="#971F1F"), opacity=0.6)


# bubbles at the edges are not clipped
fig_canton_bubble.update_xaxes(range=[x_min * 0.96, x_max * 1.04])
fig_canton_bubble.update_yaxes(range=[max(0, y_min - 0.8), y_max + 0.8])

fig_canton_bubble.show()

Interpretation of the canton bubble plot

The chart compares Migros market coverage and purchasing power across Swiss cantons:

X-axis: Average income per taxpayer
Y-axis: Migros stores per 100,000 residents
Bubble size: Canton population
Bubble colour: External competitors per 100,000 residents
(Coop + Aldi + Lidl; Denner is excluded because it belongs to Migros Group)

Main findings:
Basel-Stadt (BS) has the highest Migros coverage per 100,000 residents and relatively high income, indicating a strong and affluent Migros presence.
Lucerne (LU) also combines above-average income with strong Migros coverage.
Zürich (ZH) has the largest population, shown by the largest bubble. However, its Migros coverage is slightly below the median, despite relatively high purchasing power. This makes Zürich strategically important, although postcode-level analysis is needed to identify actual gaps.
Schwyz (SZ) stands out as a potential opportunity: it has high average income but relatively low Migros coverage.
Aargau (AG) also has below-median Migros coverage and a large population, making it worth investigating further even though its income is closer to the middle.
Zug (ZG) has the highest average income and above-average Migros coverage. It represents a strong affluent market rather than an obvious coverage gap.
Cantons in the upper-left quadrant have strong Migros coverage but lower-than-median income. These markets may require coverage review rather than immediate expansion.
Cantons in the lower-left quadrant combine lower income and lower Migros coverage, so they are generally lower-priority expansion markets.
Overall conclusion

The plot suggests that Schwyz, Zürich and Aargau deserve closer investigation. They combine meaningful population or purchasing power with comparatively weak Migros coverage. However, canton-level results are only a screening tool—the final store recommendations should be based on postcode-level population, income, existing stores and competitor presence.
 

# moving  from canton-level screening to postcode-level analysis for Zürich (ZH), Schwyz (SZ), and Aargau (AG).

In [46]:
# First, focus on postcodes that currently have no Migros supermarket.
# Create the postcode opportunity dataset
# Select complete postcodes in Zürich, Schwyz and Aargau
selected_cantons = ["ZH", "SZ", "AG"]

postcode_opportunity_df = scoring_df[
    scoring_df["primary_canton"].isin(selected_cantons) &
    (scoring_df["migros_count"] == 0) #only where Migros stores is not there
].copy()

# Create a clear label because one locality can contain multiple postcodes
postcode_opportunity_df["location_label"] = postcode_opportunity_df["locality_name"] + " (" + postcode_opportunity_df["postal_code"] + ")"

# Sort by population
postcode_opportunity_df = postcode_opportunity_df.sort_values("population", ascending=False)

print("Postcodes without Migros:", len(postcode_opportunity_df))
print("Population without Migros:", f"{postcode_opportunity_df['population'].sum():,}")

display(
    postcode_opportunity_df[
        [
            "postal_code",
            "locality_name",
            "municipality_name",
            "primary_canton",
            "population",
            "avg_income_per_taxpayer",
            "migros_count",
            "coop_count",
            "denner_count",
            "aldi_count",
            "lidl_count",
            "external_competitor_count"
        ]
    ].head(20)
)

Postcodes without Migros: 394
Population without Migros: 891,162


,postal_code,locality_name,municipality_name,primary_canton,population,avg_income_per_taxpayer,migros_count,coop_count,denner_count,aldi_count,lidl_count,external_competitor_count
2916,8854,Siebnen,"Galgenen, Schübelbach, Wangen (SZ)",SZ,11369,76149.128060,0,1,1,1,1,3
2887,8802,Kilchberg ZH,Kilchberg (ZH),ZH,9571,141795.142249,0,1,1,0,0,1
2822,8623,Wetzikon ZH,"Pfäffikon, Wetzikon (ZH)",ZH,9092,68251.227040,0,0,1,0,0,0
1588,5036,Oberentfelden,Oberentfelden,AG,9061,63550.554636,0,1,0,1,0,2
1517,4663,Aarburg,"Aarburg, Boningen, Olten",AG,9049,63019.293937,0,0,0,0,0,0
2649,8305,Dietlikon,Dietlikon,ZH,8033,77314.574859,0,2,0,1,0,3
2914,8852,Altendorf,Altendorf,SZ,7533,111522.063033,0,1,1,0,1,2
1686,5436,Würenlos,Würenlos,AG,7020,85925.606709,0,1,0,0,0,1
2573,8157,Dielsdorf,Dielsdorf,ZH,6817,71141.154995,0,0,0,1,0,1
2545,8107,Buchs ZH,"Buchs (ZH), Otelfingen",ZH,6816,74230.510768,0,1,1,0,0,1


In [47]:
# compare the three cantons
postcode_canton_summary = postcode_opportunity_df.groupby("primary_canton", as_index=False).agg(postcodes_without_migros=("postal_code", "count"), population_without_migros=("population", "sum"), average_income=("avg_income_per_taxpayer", "mean"), competitor_stores=("external_competitor_count", "sum"))

display(postcode_canton_summary)

,primary_canton,postcodes_without_migros,population_without_migros,average_income,competitor_stores
0,AG,198,392834,71747.686427,33
1,SZ,36,87112,79105.398536,10
2,ZH,160,411216,80233.813670,39


In [48]:
# plottin only top 10 postcodes per canton
# Select the top 10 most populated postcodes without Migros in each canton
from plotly.subplots import make_subplots

top_postcodes_by_canton = postcode_opportunity_df.sort_values(["primary_canton", "population"], ascending=[True, False]).groupby("primary_canton").head(10).copy()

top_postcodes_by_canton["location_label"] = top_postcodes_by_canton["locality_name"] + " (" + top_postcodes_by_canton["postal_code"] + ")"

In [49]:
# Use the same income colour range in all three charts so that their colours are comparable:
canton_names = {"ZH": "Zürich", "SZ": "Schwyz", "AG": "Aargau"}
income_colors = [[0.0, "#FFF7EC"], [0.25, "#FDD49E"], [0.50, "#FC8D59"], [0.75, "#D7301F"], [1.0, "#7F0000"]]

income_min = top_postcodes_by_canton["avg_income_per_taxpayer"].min()
income_max = top_postcodes_by_canton["avg_income_per_taxpayer"].max()

In [50]:
# Create one row and three columns
fig_postcode_facets = make_subplots(
    rows=1,
    cols=3,
    shared_yaxes=True,
    horizontal_spacing=0.06,
    subplot_titles=["Zürich (ZH)", "Schwyz (SZ)", "Aargau (AG)"]
)

In [51]:
# Add one vertical bar chart for each canton
for column, canton in enumerate(["ZH", "SZ", "AG"], start=1):
    
    # Sort from highest to lowest population
    canton_postcodes = top_postcodes_by_canton[top_postcodes_by_canton["primary_canton"] == canton].sort_values("population", ascending=False)

    fig_postcode_facets.add_trace(
        go.Bar(
            x=canton_postcodes["postal_code"],
            y=canton_postcodes["population"],
            marker=dict(
                color=canton_postcodes["avg_income_per_taxpayer"],
                coloraxis="coloraxis",
                line=dict(color="white", width=1)
            ),
            text=canton_postcodes["population"],
            texttemplate="%{text:,.0f}",
            textposition="outside",
            cliponaxis=False,
            customdata=canton_postcodes[
                [
                    "locality_name",
                    "municipality_name",
                    "avg_income_per_taxpayer",
                    "external_competitor_count",
                    "coop_count",
                    "aldi_count",
                    "lidl_count",
                    "denner_count"
                ]
            ].to_numpy(),
            hovertemplate=(
                "<b>%{customdata[0]} (%{x})</b><br>"
                "Municipality: %{customdata[1]}<br>"
                "Population: %{y:,.0f}<br>"
                "Average income: CHF %{customdata[2]:,.0f}<br>"
                "External competitors: %{customdata[3]:,.0f}<br>"
                "Coop: %{customdata[4]:,.0f}<br>"
                "Aldi: %{customdata[5]:,.0f}<br>"
                "Lidl: %{customdata[6]:,.0f}<br>"
                "Denner: %{customdata[7]:,.0f}"
                "<extra></extra>"
            ),
            showlegend=False
        ),
        row=1,
        col=column
    )

In [52]:
# format the plot
fig_postcode_facets.update_layout(
    title=dict(
        text="Largest Postcodes Without Migros in Zürich, Schwyz and Aargau",
        x=0.5,
        xanchor="center",
        font=dict(size=22, color="white")
    ),
    coloraxis=dict(
        colorscale=income_colors,
        cmin=income_min,
        cmax=income_max,
        colorbar=dict(
            title="Average income<br>per taxpayer",
            tickprefix="CHF ",
            tickformat=",.0f",
            tickfont=dict(color="white"),
            title_font=dict(color="white")
        )
    ),
    paper_bgcolor="#2B2B2B",
    plot_bgcolor="#3A3A3A",
    font=dict(color="white"),
    template="plotly_dark",
    height=550,
    width=1350,
    margin=dict(l=80, r=150, t=100, b=90),
    bargap=0.20
)

fig_postcode_facets.update_xaxes(
    title_text="Postcode",
    title_font=dict(size=18, color="white"),
    type="category",
    tickangle=-45,
    tickfont=dict(size=14, color="white"),
    showline=True,
    linewidth=2,
    linecolor="#FFFFFF",
    mirror=True,
    gridcolor="#555555",
    zeroline=False
)

fig_postcode_facets.update_yaxes(
    title_font=dict(size=18, color="white"),
    tickformat=",",
    tickfont=dict(size=16, color="white"),
    showline=True,
    linewidth=2,
    linecolor="#FFFFFF",
    mirror=True,
    gridcolor="#555555",
    zeroline=False
)

fig_postcode_facets.update_yaxes(title_text="Population", row=1, col=1)
fig_postcode_facets.update_yaxes(title_text="", row=1, col=2)
fig_postcode_facets.update_yaxes(title_text="", row=1, col=3)


fig_postcode_facets.add_shape(type="line", x0=0.325, x1=0.325, y0=0, y1=1, xref="paper", yref="paper", line=dict(color="#FFFFFF", width=3))
fig_postcode_facets.add_shape(type="line", x0=0.675, x1=0.675, y0=0, y1=1, xref="paper", yref="paper", line=dict(color="#FFFFFF", width=3))

fig_postcode_facets.for_each_annotation(lambda annotation: annotation.update(font=dict(size=16, color="white")))

fig_postcode_facets.show()

Summary of postcode-level analysis

The chart shows the ten most populated postcodes without a Migros in Zürich, Schwyz and Aargau. Bar height represents population, while darker red indicates higher average income per taxpayer.

Zürich: Postcodes 8802 and 8623 have the largest uncovered populations, with approximately 9,571 and 9,092 residents. Postcodes 8802 and 8704 combine relatively large populations with especially high income, making them strong candidates for further investigation.
Schwyz: Postcode 8854 is the clearest population gap, with approximately 11,369 residents, the highest value across all three cantons. Postcode 8852 follows with 7,533 residents. Postcode 8807 has a smaller population but particularly high income, so it may represent a smaller affluent-market opportunity.
Aargau: Postcodes 5036 and 4663 lead with approximately 9,061 and 9,049 residents. However, most of the leading Aargau postcodes have lighter colours, indicating lower average income than the strongest candidates in Zürich and Schwyz.

Main conclusion:

8854 in Schwyz has the largest population without a Migros.
8802 and 8704 in Zürich appear especially attractive because they combine meaningful population with high purchasing power.
5036 and 4663 in Aargau offer large population potential, although purchasing power appears comparatively lower.


# The next comparison should include competitor presence and distance to the nearest existing Migros before selecting the best new-store locations.

The next comparison should evaluate the shortlisted postcodes using:

Population
Income
External competitors
Distance to the nearest existing Migros - we need to calculate from postcode coordinates

In [53]:
#Calculate distance to the nearest Migros postcode
from sklearn.neighbors import BallTree

In [54]:
# Select postcode centres that already contain at least one Migros
migros_postcodes_df = scoring_df[scoring_df["migros_count"] > 0].copy()

In [55]:
# Build a geographical search tree using Migros postcode coordinates
migros_coordinates_rad = np.radians(migros_postcodes_df[["postcode_latitude", "postcode_longitude"]])
migros_tree = BallTree(migros_coordinates_rad, metric="haversine")

In [56]:
# Find the nearest Migros postcode for every opportunity postcode
opportunity_coordinates_rad = np.radians(postcode_opportunity_df[["postcode_latitude", "postcode_longitude"]])
nearest_distance, nearest_index = migros_tree.query(opportunity_coordinates_rad, k=1)

In [57]:
# Convert the angular distance to kilometres
postcode_opportunity_df["nearest_migros_distance_km"] = nearest_distance.flatten() * 6371

In [58]:
# Store the postcode of the nearest existing Migros
postcode_opportunity_df["nearest_migros_postcode"] = migros_postcodes_df.iloc[nearest_index.flatten()]["postal_code"].to_numpy()

In [59]:
comparison_columns = ["postal_code", "locality_name", "primary_canton", "population", "avg_income_per_taxpayer", "external_competitor_count", "nearest_migros_postcode", "nearest_migros_distance_km"]
postcode_comparison_df = postcode_opportunity_df.sort_values(["population", "avg_income_per_taxpayer"], ascending=[False, False]).copy()
display(postcode_comparison_df[comparison_columns].head(30).style.format({"population": "{:,.0f}", "avg_income_per_taxpayer": "CHF {:,.0f}", "nearest_migros_distance_km": "{:.2f} km"}))

,postal_code,locality_name,primary_canton,population,avg_income_per_taxpayer,external_competitor_count,nearest_migros_postcode,nearest_migros_distance_km
2916,8854,Siebnen,SZ,"11,369","CHF 76,149",3,8853,3.03 km
2887,8802,Kilchberg ZH,ZH,"9,571","CHF 141,795",1,8803,1.57 km
2822,8623,Wetzikon ZH,ZH,"9,092","CHF 68,251",0,8620,2.04 km
1588,5036,Oberentfelden,AG,"9,061","CHF 63,551",2,5035,1.44 km
1517,4663,Aarburg,AG,"9,049","CHF 63,019",0,4852,1.90 km
2649,8305,Dietlikon,ZH,"8,033","CHF 77,315",3,8304,1.77 km
2914,8852,Altendorf,SZ,"7,533","CHF 111,522",2,8853,2.62 km
1686,5436,Würenlos,AG,"7,020","CHF 85,926",1,8957,2.46 km
2573,8157,Dielsdorf,ZH,"6,817","CHF 71,141",1,8155,1.90 km
2545,8107,Buchs ZH,ZH,"6,816","CHF 74,231",1,8955,3.72 km


A strong new-store candidate should ideally have:
High population
High income
A large distance from the nearest Migros
A manageable number of external competitors

also:
Few competitors may indicate an underserved market.
Many competitors may indicate proven supermarket demand but stronger competition.

Therefore, competitor count should not automatically be treated as entirely negative.

The distance here is between postcode centres, distance from exact Migros store. It is suitable for initial screening, but final recommendations should use the original store coordinates.

In [60]:
# Update the top-ten data for the next plot
top_postcodes_by_canton = postcode_opportunity_df.sort_values(["primary_canton", "population"], ascending=[True, False]).groupby("primary_canton").head(10).copy()
top_postcodes_by_canton.head()

,postal_code,population,avg_income_per_taxpayer,municipality_name,canton_code,aldi_count,coop_count,denner_count,lidl_count,migros_count,external_competitor_count,migros_group_count,postcode_latitude,postcode_longitude,locality_name,complete_location_data,has_migros,population_per_migros,primary_canton,location_label,nearest_migros_distance_km,nearest_migros_postcode
1588,5036,9061,63550.554636,Oberentfelden,AG,1,1,0,0,0,2,0,47.356506,8.043692,Oberentfelden,True,False,9061.0,AG,Oberentfelden (5036),1.435505,5035
1517,4663,9049,63019.293937,"Aarburg, Boningen, Olten","AG, SO",0,0,0,0,0,0,0,47.315838,7.891277,Aarburg,True,False,9049.0,AG,Aarburg (4663),1.903732,4852
1686,5436,7020,85925.606709,Würenlos,AG,0,1,0,0,0,1,0,47.441429,8.362659,Würenlos,True,False,7020.0,AG,Würenlos (5436),2.459856,8957
1617,5102,6303,70845.331784,Rupperswil,AG,0,0,1,0,0,0,1,47.401320,8.128204,Rupperswil,True,False,6303.0,AG,Rupperswil (5102),3.195422,5103
1372,4303,5993,74201.023232,"Augst, Giebenach, Kaiseraugst, Rheinfelden","AG, BL",0,1,1,0,0,1,1,47.536928,7.737454,Kaiseraugst,True,False,5993.0,AG,Kaiseraugst (4303),3.457749,4414


In [61]:
# Next, combine population, income, distance and competition into an opportunity score to rank the postcodes.
# lets Normalize the variables (0 to 1) so that population, income and distance can be combined.
from sklearn.preprocessing import MinMaxScaler


In [62]:
scaler = MinMaxScaler()
postcode_opportunity_df[["population_score", "income_score", "distance_score", "competitor_level"]] = scaler.fit_transform(postcode_opportunity_df[["population", "avg_income_per_taxpayer", "nearest_migros_distance_km", "external_competitor_count"]])

# For population, income and distance, a higher value is desirable. For competitors, a lower value is preferable
postcode_opportunity_df["low_competition_score"] = 1 - postcode_opportunity_df["competitor_level"]


In [63]:
# Calculate the opportunity score
# initial weights:
# Population: 40%
# Income: 25%
# Distance from Migros: 25%
# Low competition: 10%
postcode_opportunity_df["opportunity_score"] = (
    postcode_opportunity_df["population_score"] * 0.40 +
    postcode_opportunity_df["income_score"] * 0.25 +
    postcode_opportunity_df["distance_score"] * 0.25 +
    postcode_opportunity_df["low_competition_score"] * 0.10
)

#Convert the score to a more readable 0–100 scale
postcode_opportunity_df["opportunity_score"] = postcode_opportunity_df["opportunity_score"] * 100

#rank the postcodes
postcode_ranking_df = postcode_opportunity_df.sort_values("opportunity_score", ascending=False).reset_index(drop=True)
postcode_ranking_df["rank"] = postcode_ranking_df.index + 1

ranking_columns = ["rank", "postal_code", "locality_name", "primary_canton", "population", "avg_income_per_taxpayer", "external_competitor_count", "nearest_migros_postcode", "nearest_migros_distance_km", "opportunity_score"]

display(
    postcode_ranking_df[ranking_columns].head(20).style.format({
        "population": "{:,.0f}",
        "avg_income_per_taxpayer": "CHF {:,.0f}",
        "nearest_migros_distance_km": "{:.2f} km",
        "opportunity_score": "{:.1f}"
    })
)

,rank,postal_code,locality_name,primary_canton,population,avg_income_per_taxpayer,external_competitor_count,nearest_migros_postcode,nearest_migros_distance_km,opportunity_score
0,1,8802,Kilchberg ZH,ZH,"9,571","CHF 141,795",1,8803,1.57 km,61.9
1,2,8854,Siebnen,SZ,"11,369","CHF 76,149",3,8853,3.03 km,56.6
2,3,8623,Wetzikon ZH,ZH,"9,092","CHF 68,251",0,8620,2.04 km,54.8
3,4,4663,Aarburg,AG,"9,049","CHF 63,019",0,4852,1.90 km,53.6
4,5,8907,Wettswil,ZH,"5,419","CHF 110,417",0,8041,3.07 km,50.4
5,6,8704,Herrliberg,ZH,"6,792","CHF 156,484",2,8703,1.41 km,50.3
6,7,8852,Altendorf,SZ,"7,533","CHF 111,522",2,8853,2.62 km,50.1
7,8,8932,Mettmenstetten,ZH,"5,841","CHF 87,008",0,8910,3.51 km,49.9
8,9,8165,Schöfflisdorf,ZH,"4,202","CHF 80,967",0,8155,5.83 km,49.6
9,10,8842,Unteriberg,SZ,"2,093","CHF 62,697",0,6436,9.41 km,49.5


In [64]:
# Plot the 15 strongest candidates
top_15_opportunities = postcode_ranking_df.head(15).sort_values("opportunity_score", ascending=True).copy()

#top_15_opportunities["location_label"] = top_15_opportunities["locality_name"] + " (" + top_15_opportunities["postal_code"] + ")"


In [65]:
# Select the top 15 ranked postcode opportunities
# Descending rank order is needed because Plotly draws horizontal bars from bottom to top
#top_15_opportunities = postcode_ranking_df.nsmallest(15, "rank").sort_values("rank", ascending=False).copy()

# Create a label containing rank, locality and postcode
top_15_opportunities["rank_label"] = "#" + top_15_opportunities["rank"].astype(str) + "  " + top_15_opportunities["locality_name"] + " (" + top_15_opportunities["postal_code"] + ")"

# Create the ranked opportunity chart
fig_opportunity_ranking = px.bar(
    top_15_opportunities,
    x="opportunity_score",
    y="rank_label",
    orientation="h",
    color="primary_canton",
    color_discrete_map={
        "ZH": "#E41A1C",
        "SZ": "#377EB8",
        "AG": "#4DAF4A"
    },
    text="opportunity_score",
    custom_data=[
        "rank",
        "primary_canton",
        "population",
        "avg_income_per_taxpayer",
        "nearest_migros_distance_km",
        "nearest_migros_postcode",
        "external_competitor_count"
    ],
    title="Top 15 Ranked Postcode Opportunities for a New Migros"
)

# Format bar labels and hover information
fig_opportunity_ranking.update_traces(
    texttemplate="%{text:.1f}",
    textposition="outside",
    cliponaxis=False,
    hovertemplate=(
        "<b>%{y}</b><br>"
        "Rank: %{customdata[0]:.0f}<br>"
        "Canton: %{customdata[1]}<br>"
        "Population: %{customdata[2]:,.0f}<br>"
        "Average income: CHF %{customdata[3]:,.0f}<br>"
        "Nearest Migros distance: %{customdata[4]:.2f} km<br>"
        "Nearest Migros postcode: %{customdata[5]}<br>"
        "External competitors: %{customdata[6]:,.0f}<br>"
        "Opportunity score: %{x:.1f}"
        "<extra></extra>"
    )
)

# Apply the same dark theme as the previous facet plots
fig_opportunity_ranking.update_layout(
    title=dict(
        text="Top 15 Ranked Postcode Opportunities for a New Migros",
        x=0.5,
        xanchor="center",
        font=dict(size=23, color="white")
    ),
    xaxis=dict(
        title=dict(text="Opportunity score", font=dict(size=17, color="white")),
        tickfont=dict(size=13, color="white"),
        showline=True,
        linewidth=2,
        linecolor="#FFFFFF",
        mirror=True,
        gridcolor="#555555",
        zeroline=False
    ),
    yaxis=dict(
        title="",
        tickfont=dict(size=13, color="white"),
        categoryorder="array",
        categoryarray=top_15_opportunities["rank_label"].tolist(),
        showline=True,
        linewidth=2,
        linecolor="#FFFFFF",
        mirror=True,
        gridcolor="#555555",
        zeroline=False
    ),
    legend=dict(
        title=dict(text="Canton", font=dict(size=14, color="white")),
        font=dict(size=13, color="white"),
        bgcolor="#2B2B2B",
        bordercolor="#FFFFFF",
        borderwidth=1
    ),
    paper_bgcolor="#2B2B2B",
    plot_bgcolor="#3A3A3A",
    font=dict(color="white"),
    template="plotly_dark",
    height=750,
    width=1150,
    margin=dict(l=280, r=120, t=100, b=80),
    bargap=0.22
)

fig_opportunity_ranking.show()

based on the above graph:
1.8802 Kilchberg ZH
2. 8854 Siebnen SZ
are the top 2 candidates with high opertunity to open a new Migros store.
lets deeply evaluate these 2 postal codes.
The current score only considers the candidate postcode itself. It does not yet measure the surrounding catchment area. This matters because Kilchberg’s nearest Migros postcode is only 1.57 km away, which may substantially reduce the real opportunity.

# Deep analysis 1: Kilchberg 8802

In [66]:
# The following code calculates the distance from 8802 to every other postcode and summarizes the population and stores within 2, 5 and 10 km.
# Select the candidate postcode
candidate_postcode = "8802"

candidate = scoring_df.loc[scoring_df["postal_code"] == candidate_postcode].iloc[0]

print("Candidate:", candidate["locality_name"], f"({candidate_postcode})")
print("Canton:", candidate["primary_canton"])
print("Population:", f"{candidate['population']:,.0f}")
print("Average income:", f"CHF {candidate['avg_income_per_taxpayer']:,.0f}")
print("Migros stores:", candidate["migros_count"])
print("External competitors:", candidate["external_competitor_count"])


Candidate: Kilchberg ZH (8802)
Canton: ZH
Population: 9,571
Average income: CHF 141,795
Migros stores: 0
External competitors: 1


In [67]:
# Calculate distance to surrounding postcodes
# Convert the candidate coordinates to radians
candidate_latitude = np.radians(candidate["postcode_latitude"])
candidate_longitude = np.radians(candidate["postcode_longitude"])

# Convert all postcode coordinates to radians
postcode_latitudes = np.radians(scoring_df["postcode_latitude"])
postcode_longitudes = np.radians(scoring_df["postcode_longitude"])

# Calculate the Haversine distance between 8802 and every postcode
latitude_difference = postcode_latitudes - candidate_latitude
longitude_difference = postcode_longitudes - candidate_longitude

haversine_value = (
    np.sin(latitude_difference / 2) ** 2 +
    np.cos(candidate_latitude) *
    np.cos(postcode_latitudes) *
    np.sin(longitude_difference / 2) ** 2
)

scoring_df["distance_from_8802_km"] = 6371 * 2 * np.arctan2(np.sqrt(haversine_value), np.sqrt(1 - haversine_value))

In [68]:
# Display the closest postcodes
nearby_8802_df = scoring_df.sort_values("distance_from_8802_km").copy()

display(
    nearby_8802_df[
        [
            "postal_code",
            "locality_name",
            "primary_canton",
            "distance_from_8802_km",
            "population",
            "avg_income_per_taxpayer",
            "migros_count",
            "denner_count",
            "external_competitor_count"
        ]
    ].head(20).style.format({
        "distance_from_8802_km": "{:.2f} km",
        "population": "{:,.0f}",
        "avg_income_per_taxpayer": "CHF {:,.0f}"
    })
)

,postal_code,locality_name,primary_canton,distance_from_8802_km,population,avg_income_per_taxpayer,migros_count,denner_count,external_competitor_count
2887,8802,Kilchberg ZH,ZH,0.00 km,"9,571","CHF 141,795",0,1,1
2888,8803,Rüschlikon,ZH,1.57 km,"6,454","CHF 135,647",1,0,1
2563,8134,Adliswil,ZH,1.93 km,"19,833","CHF 82,969",1,2,4
2525,8038,Zürich,ZH,2.16 km,"18,395","CHF 77,136",1,1,4
2526,8041,Zürich,ZH,2.25 km,"9,648","CHF 77,151",1,0,1
2839,8702,Zollikon,ZH,3.02 km,"8,360","CHF 142,354",1,0,1
2838,8700,Küsnacht ZH,ZH,3.37 km,"14,482","CHF 155,644",1,0,2
2886,8800,Thalwil,ZH,3.66 km,"16,154","CHF 100,808",1,2,1
2522,8008,Zürich,ZH,3.82 km,"17,495","CHF 77,136",3,0,4
2517,8002,Zürich,ZH,4.27 km,"9,992","CHF 77,136",1,0,1


In [69]:
# Calculate 2 km, 5 km and 10 km catchment areas
catchment_results = []

for radius in [2, 5, 10]:
    catchment = nearby_8802_df[nearby_8802_df["distance_from_8802_km"] <= radius]

    catchment_results.append({
        "radius_km": radius,
        "postcodes": catchment["postal_code"].nunique(),
        "population": catchment["population"].sum(),
        "migros_stores": catchment["migros_count"].sum(),
        "denner_stores": catchment["denner_count"].sum(),
        "external_competitors": catchment["external_competitor_count"].sum()
    })

catchment_8802_summary = pd.DataFrame(catchment_results)
display(catchment_8802_summary)

,radius_km,postcodes,population,migros_stores,denner_stores,external_competitors
0,2,3,35858,2,3,6
1,5,13,148643,12,8,25
2,10,58,623367,47,42,108


In [70]:
# Plot the Kilchberg catchment comparison
catchment_8802_long = catchment_8802_summary.melt(
    id_vars="radius_km",
    value_vars=["migros_stores", "denner_stores", "external_competitors"],
    var_name="store_type",
    value_name="store_count"
)

fig_8802_catchment = px.bar(
    catchment_8802_long,
    x="radius_km",
    y="store_count",
    color="store_type",
    barmode="group",
    text="store_count",
    color_discrete_map={
        "migros_stores": "#F58220",
        "denner_stores": "#19A831",
        "external_competitors": "#377EB8"
    },
    title="Supermarket Competition Around Kilchberg 8802"
)

fig_8802_catchment.update_layout(
    title=dict(x=0.5, xanchor="center", font=dict(size=23, color="white")),
    xaxis=dict(title="Catchment radius (km)", tickvals=[2, 5, 10], gridcolor="#555555"),
    yaxis=dict(title="Number of stores", gridcolor="#555555"),
    paper_bgcolor="#2B2B2B",
    plot_bgcolor="#3A3A3A",
    font=dict(color="white"),
    template="plotly_dark",
    height=550,
    width=900
)

fig_8802_catchment.show()

Within only 2 km, the area already contains:

2 Migros stores
3 Denner stores
6 external competitors
35,858 residents

Migros Group therefore already has five stores within 2 km when Migros and Denner are combined.

In [71]:
# Calculate the store densities:
catchment_8802_summary["migros_per_100k"] = catchment_8802_summary["migros_stores"] / catchment_8802_summary["population"] * 100000
catchment_8802_summary["migros_group_per_100k"] = (catchment_8802_summary["migros_stores"] + catchment_8802_summary["denner_stores"]) / catchment_8802_summary["population"] * 100000
catchment_8802_summary["competitors_per_100k"] = catchment_8802_summary["external_competitors"] / catchment_8802_summary["population"] * 100000

display(catchment_8802_summary.round(2))

,radius_km,postcodes,population,migros_stores,denner_stores,external_competitors,migros_per_100k,migros_group_per_100k,competitors_per_100k
0,2,3,35858,2,3,6,5.58,13.94,16.73
1,5,13,148643,12,8,25,8.07,13.46,16.82
2,10,58,623367,47,42,108,7.54,14.28,17.33


Conclusion for 8802

Kilchberg ranks highly mainly because of:

Its very high average income
Its relatively large postcode population
Only one competitor recorded directly inside postcode 8802

But the catchment analysis reveals substantial nearby supermarket coverage. Therefore:

Kilchberg 8802 has strong purchasing power, but it is already well served by nearby Migros and Denner stores. It should not automatically be treated as the strongest expansion opportunity.

This demonstrates why postcode-level counts alone can be misleading.

# 2. Siebnen 8854

In [72]:
# Deep catchment analysis for Siebnen 8854
candidate_postcode = "8854"
candidate = scoring_df.loc[scoring_df["postal_code"] == candidate_postcode].iloc[0]

candidate_latitude = np.radians(candidate["postcode_latitude"])
candidate_longitude = np.radians(candidate["postcode_longitude"])
postcode_latitudes = np.radians(scoring_df["postcode_latitude"])
postcode_longitudes = np.radians(scoring_df["postcode_longitude"])

latitude_difference = postcode_latitudes - candidate_latitude
longitude_difference = postcode_longitudes - candidate_longitude

haversine_value = (
    np.sin(latitude_difference / 2) ** 2 +
    np.cos(candidate_latitude) *
    np.cos(postcode_latitudes) *
    np.sin(longitude_difference / 2) ** 2
)

scoring_df["distance_from_8854_km"] = 6371 * 2 * np.arctan2(np.sqrt(haversine_value), np.sqrt(1 - haversine_value))

nearby_8854_df = scoring_df.sort_values("distance_from_8854_km").copy()

catchment_results = []

for radius in [2, 5, 10]:
    catchment = nearby_8854_df[nearby_8854_df["distance_from_8854_km"] <= radius]

    catchment_results.append({
        "radius_km": radius,
        "postcodes": catchment["postal_code"].nunique(),
        "population": catchment["population"].sum(),
        "migros_stores": catchment["migros_count"].sum(),
        "denner_stores": catchment["denner_count"].sum(),
        "external_competitors": catchment["external_competitor_count"].sum()
    })

catchment_8854_summary = pd.DataFrame(catchment_results)

catchment_8854_summary["migros_per_100k"] = catchment_8854_summary["migros_stores"] / catchment_8854_summary["population"] * 100000
catchment_8854_summary["migros_group_per_100k"] = (catchment_8854_summary["migros_stores"] + catchment_8854_summary["denner_stores"]) / catchment_8854_summary["population"] * 100000
catchment_8854_summary["competitors_per_100k"] = catchment_8854_summary["external_competitors"] / catchment_8854_summary["population"] * 100000

print("Candidate:", candidate["locality_name"], f"({candidate_postcode})")
print("Population:", f"{candidate['population']:,.0f}")
print("Average income:", f"CHF {candidate['avg_income_per_taxpayer']:,.0f}")

display(catchment_8854_summary.round(2))

Candidate: Siebnen (8854)
Population: 11,369
Average income: CHF 76,149


,radius_km,postcodes,population,migros_stores,denner_stores,external_competitors,migros_per_100k,migros_group_per_100k,competitors_per_100k
0,2,2,15376,0,1,3,0.00,6.50,19.51
1,5,7,34064,1,3,4,2.94,11.74,11.74
2,10,23,119561,9,12,20,7.53,17.56,16.73


Interpretation of Siebnen 8854

Siebnen appears to be a much more genuine coverage gap than Kilchberg.

Within 2 km:

Population: 15,376
Migros stores: 0
Denner stores: 1
External competitors: 3
Migros Group coverage: 6.50 stores per 100,000 residents

Within 5 km:

Population: 34,064
Migros stores: only 1
Denner stores: 3
External competitors: 4
Migros coverage: 2.94 stores per 100,000 residents

The relatively low Migros coverage within 2–5 km suggests that Siebnen is underserved by Migros supermarkets.

In [73]:
# # Plot the Siebnen  catchment comparison
# Convert the Siebnen catchment summary into long format
catchment_8854_long = catchment_8854_summary.melt(
    id_vars="radius_km",
    value_vars=["migros_stores", "denner_stores", "external_competitors"],
    var_name="store_type",
    value_name="store_count"
)

# Rename the categories for clearer chart labels
catchment_8854_long["store_type"] = catchment_8854_long["store_type"].replace({
    "migros_stores": "Migros",
    "denner_stores": "Denner",
    "external_competitors": "External competitors"
})

# Create the grouped bar chart
fig_siebnen = px.bar(
    catchment_8854_long,
    x="radius_km",
    y="store_count",
    color="store_type",
    barmode="group",
    text="store_count",
    color_discrete_map={
        "Migros": "#F58220",
        "Denner": "#19A831",
        "External competitors": "#377EB8"
    },
    title="Supermarket Coverage Around Siebnen 8854"
)

# Format the bars
fig_siebnen.update_traces(
    textposition="outside",
    cliponaxis=False,
    hovertemplate=(
        "<b>%{fullData.name}</b><br>"
        "Radius: %{x} km<br>"
        "Stores: %{y:,.0f}"
        "<extra></extra>"
    )
)

# Apply the dark theme
fig_siebnen.update_layout(
    title=dict(
        x=0.5,
        xanchor="center",
        font=dict(size=23, color="white")
    ),
    xaxis=dict(
        title=dict(text="Catchment radius (km)", font=dict(size=17)),
        tickvals=[2, 5, 10],
        tickfont=dict(size=13),
        showline=True,
        linewidth=2,
        linecolor="white",
        mirror=True,
        gridcolor="#555555",
        zeroline=False
    ),
    yaxis=dict(
        title=dict(text="Number of stores", font=dict(size=17)),
        tickfont=dict(size=13),
        showline=True,
        linewidth=2,
        linecolor="white",
        mirror=True,
        gridcolor="#555555",
        zeroline=False
    ),
    legend=dict(
        title="Store type",
        bgcolor="#2B2B2B",
        bordercolor="white",
        borderwidth=1
    ),
    paper_bgcolor="#2B2B2B",
    plot_bgcolor="#3A3A3A",
    font=dict(color="white"),
    template="plotly_dark",
    height=550,
    width=950,
    bargap=0.20
)

fig_siebnen.show()

In [74]:
#plot Siebnen versus Kilchberg

# Create copies and identify each candidate
kilchberg_comparison = catchment_8802_summary.copy()
kilchberg_comparison["location"] = "Kilchberg 8802"

siebnen_comparison = catchment_8854_summary.copy()
siebnen_comparison["location"] = "Siebnen 8854"

# Combine the two catchment summaries
candidate_comparison_df = pd.concat(
    [kilchberg_comparison, siebnen_comparison],
    ignore_index=True
)

# Create three comparison panels
fig_candidate_comparison = make_subplots(
    rows=1,
    cols=3,
    horizontal_spacing=0.10,
    subplot_titles=[
        "Catchment Population",
        "Migros Group Coverage",
        "External Competition"
    ]
)

location_colors = {
    "Kilchberg 8802": "#377EB8",
    "Siebnen 8854": "#F58220"
}

# Add one line for each candidate
for location in ["Kilchberg 8802", "Siebnen 8854"]:

    location_data = candidate_comparison_df[
        candidate_comparison_df["location"] == location
    ].sort_values("radius_km")

    fig_candidate_comparison.add_trace(
        go.Scatter(
            x=location_data["radius_km"],
            y=location_data["population"],
            mode="lines+markers+text",
            name=location,
            legendgroup=location,
            marker=dict(size=11, color=location_colors[location]),
            line=dict(width=3, color=location_colors[location]),
            text=location_data["population"],
            texttemplate="%{text:,.0f}",
            textposition="top center",
            hovertemplate="<b>" + location + "</b><br>Radius: %{x} km<br>Population: %{y:,.0f}<extra></extra>"
        ),
        row=1,
        col=1
    )

    fig_candidate_comparison.add_trace(
        go.Scatter(
            x=location_data["radius_km"],
            y=location_data["migros_group_per_100k"],
            mode="lines+markers+text",
            name=location,
            legendgroup=location,
            showlegend=False,
            marker=dict(size=11, color=location_colors[location]),
            line=dict(width=3, color=location_colors[location]),
            text=location_data["migros_group_per_100k"],
            texttemplate="%{text:.1f}",
            textposition="top center",
            hovertemplate="<b>" + location + "</b><br>Radius: %{x} km<br>Migros Group per 100K: %{y:.2f}<extra></extra>"
        ),
        row=1,
        col=2
    )

    fig_candidate_comparison.add_trace(
        go.Scatter(
            x=location_data["radius_km"],
            y=location_data["competitors_per_100k"],
            mode="lines+markers+text",
            name=location,
            legendgroup=location,
            showlegend=False,
            marker=dict(size=11, color=location_colors[location]),
            line=dict(width=3, color=location_colors[location]),
            text=location_data["competitors_per_100k"],
            texttemplate="%{text:.1f}",
            textposition="top center",
            hovertemplate="<b>" + location + "</b><br>Radius: %{x} km<br>Competitors per 100K: %{y:.2f}<extra></extra>"
        ),
        row=1,
        col=3
    )

# Format all x-axes
fig_candidate_comparison.update_xaxes(
    title_text="Radius (km)",
    tickvals=[2, 5, 10],
    showline=True,
    linewidth=2,
    linecolor="white",
    mirror=True,
    gridcolor="#555555",
    zeroline=False
)

# Format the individual y-axes
fig_candidate_comparison.update_yaxes(title_text="Population", tickformat=",", row=1, col=1)
fig_candidate_comparison.update_yaxes(title_text="Stores per 100K", row=1, col=2)
fig_candidate_comparison.update_yaxes(title_text="Stores per 100K", row=1, col=3)

# Apply the shared dark theme
fig_candidate_comparison.update_layout(
    title=dict(
        text="Catchment Comparison: Siebnen versus Kilchberg",
        x=0.5,
        xanchor="center",
        font=dict(size=23, color="white")
    ),
    legend=dict(
        title="Candidate",
        orientation="h",
        x=0.5,
        xanchor="center",
        y=1.08,
        yanchor="bottom",
        bgcolor="#2B2B2B",
        bordercolor="white",
        borderwidth=1
    ),
    paper_bgcolor="#2B2B2B",
    plot_bgcolor="#3A3A3A",
    font=dict(color="white"),
    template="plotly_dark",
    height=600,
    width=1350,
    margin=dict(l=80, r=60, t=140, b=80)
)

fig_candidate_comparison.for_each_annotation(lambda annotation: annotation.update(font=dict(size=16, color="white")))

fig_candidate_comparison.show()

Business interpretation

Kilchberg

Extremely high purchasing power
Larger surrounding population
But already strongly served by Migros and Denner
Greater risk of cannibalising existing Migros Group stores

Siebnen

Lower purchasing power than Kilchberg, but still reasonable
Higher postcode population
No Migros within 2 km
Only one Migros within 5 km
Existing external competitors suggest supermarket demand is already present
Lower risk of cannibalising another Migros store
Conclusion

Siebnen 8854 is the stronger expansion candidate. Kilchberg’s original rank was driven heavily by its exceptional income, but the catchment analysis shows that it is already well covered. Siebnen combines a larger postcode population with substantially weaker nearby Migros coverage.

In [75]:
# Save all Plotly figures created in notebook 07
import os

os.environ["BROWSER_PATH"] = "/home/lavanya/.local/share/choreographer/deps/chrome-linux64/chrome"
#from pathlib import Path

#plot_folder = Path("plots")
#plot_folder.mkdir(parents=True, exist_ok=True)

figure_variables = {
    "01_supermarket_locations_by_brand": "fig_brand_counts",
    "02_largest_population_centres_without_migros": "fig_population_gaps",
    "03_migros_stores_by_canton": "fig_migros_canton",
    "04_canton_market_coverage": "fig_canton_bubble",
    "05_postcodes_without_migros_by_canton": "fig_postcode_facets",
    "06_top_postcode_opportunities": "fig_opportunity_ranking",
    "07_kilchberg_catchment": "fig_8802_catchment",
    "08_siebnen_catchment": "fig_siebnen",
    "09_siebnen_kilchberg_comparison": "fig_candidate_comparison"
}

# Include only figures that currently exist in the notebook
figures_to_save = {
    filename: globals()[variable_name]
    for filename, variable_name in figure_variables.items()
    if variable_name in globals()
}

missing_figures = [
    variable_name
    for variable_name in figure_variables.values()
    if variable_name not in globals()
]

# Save every available figure as an interactive standalone HTML file
for filename, figure in figures_to_save.items():
    figure.write_html(
        plot_folder / f"{filename}.html",
        include_plotlyjs=True,
        full_html=True
    )

print(f"Saved {len(figures_to_save)} interactive HTML figures in '{plot_folder}'.")

if missing_figures:
    print("Figures not saved because their cells were not executed:")
    print(missing_figures)

# Try saving static PNG versions
try:
    for filename, figure in figures_to_save.items():
        figure.write_image(
            plot_folder / f"{filename}.png",
            scale=2
        )

    print(f"Saved {len(figures_to_save)} PNG figures in '{plot_folder}'.")

except Exception as error:
    print("\nInteractive HTML files were saved successfully.")
    print("PNG export was skipped because Chrome/Kaleido is unavailable.")
    print("Run 'plotly_get_chrome' in the terminal and execute this cell again.")
    print("PNG error:", error)

Saved 9 interactive HTML figures in 'plots'.
Saved 9 PNG figures in 'plots'.
